# Perplexity Analysis — TAT German

This notebook runs the full statistical analysis and visualization pipeline.
It loops over three analysis settings (with outliers, without outliers, matched sample)
and produces the same set of plots and statistics for each.

In [ ]:
import sys
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# Make sure project root is on the path
sys.path.insert(0, os.path.abspath('.'))

from services.analysis_service import analysis_service, load_metadata, build_person_df
from services.plotting_service import plot_setting, plot_cross_setting, plot_advanced
from functions.analysis.statistics import (
    CLINICAL_VARS,
    group_comparison,
    clinical_correlations,
    residualize_ppl,
    nested_regression_models,
    steigers_z,
    compute_roc,
    logistic_cv_auc,
)
from functions.plotting.plots import (
    plot_roc_curves,
    plot_metrics_heatmap,
)

## 1. Configuration

In [ ]:
# Paths
SENTENCE_CSV = "/zi/home/esra.lenz/Documents/00_HITKIP/11_normative/mnm_ppl/data/TAT_german_utterances_with_perplexities.csv"
METADATA_CSV = "data/00_raw_data/Demografics_Speech_FOR2107.csv"
FIGURE_DIR = "figures"

# Model prefix for column names
MODEL = "qwen"

# Primary perplexity variable
PPL_VAR = f"{MODEL}_ppl_median"
WC_VAR = f"{MODEL}_total_words"

# Clinical variables
clinical_vars = CLINICAL_VARS
print(f"Clinical variables: {clinical_vars}")

## 2. Load data & prepare three analysis settings

In [ ]:
df, settings = analysis_service(
    sentence_csv=SENTENCE_CSV,
    metadata_csv=METADATA_CSV,
    model_name=MODEL,
)

print(f"\nFull dataset: {len(df)} subjects")
print(f"Patients: {(df['is_patient'] == 1).sum()}, Controls: {(df['is_patient'] == 0).sum()}")

In [ ]:
df.head()

## 3. Loop: Group comparison & clinical correlations for all 3 settings

In [ ]:
# Map each setting to the PPL variable it should use
setting_ppl_vars = {
    "with_outliers": PPL_VAR,
    "without_outliers": f"{PPL_VAR}_wins",
    "matched": PPL_VAR,
}

all_results = {}
all_corr_dfs = {}

for setting_name, setting_df in settings.items():
    ppl_col = setting_ppl_vars[setting_name]
    print(f"\n{'='*60}")
    print(f"Setting: {setting_name} | PPL variable: {ppl_col}")
    print(f"{'='*60}")

    result = plot_setting(
        df=setting_df,
        ppl_var=ppl_col,
        setting_name=setting_name,
        figure_dir=FIGURE_DIR,
        clinical_vars=clinical_vars,
    )
    all_results[setting_name] = result
    all_corr_dfs[setting_name] = result["correlations"]

## 4. Cross-setting comparison

In [ ]:
plot_cross_setting(all_corr_dfs, figure_dir=FIGURE_DIR)

In [ ]:
# Summary table of all comparisons
summary_rows = []
for sname, res in all_results.items():
    row = {"Setting": sname}
    row.update(res["comparison"])
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
summary_df

## 5. Alternative metrics comparison

In [ ]:
alt_metrics = [
    f"{MODEL}_ppl_median", f"{MODEL}_ppl_mean",
    f"{MODEL}_ppl_p90", f"{MODEL}_ppl_p75",
    f"{MODEL}_ppl_std", f"{MODEL}_ppl_cv",
    f"{MODEL}_prop_high_ppl",
]

# Group comparison for each metric
alt_comp = pd.DataFrame([group_comparison(df, m) for m in alt_metrics if m in df.columns])
alt_comp

In [ ]:
# Correlation heatmap: metrics × clinical variables
patients = df[df['is_patient'] == 1].copy()
r_matrix = pd.DataFrame(index=[m for m in alt_metrics if m in df.columns], columns=clinical_vars, dtype=float)
p_matrix = r_matrix.copy()

for metric in r_matrix.index:
    for cvar in clinical_vars:
        sub = patients[[metric, cvar]].dropna()
        if len(sub) >= 5:
            r, p = spearmanr(sub[metric], sub[cvar])
            r_matrix.loc[metric, cvar] = r
            p_matrix.loc[metric, cvar] = p

plot_metrics_heatmap(
    r_matrix.astype(float),
    p_matrix.astype(float),
    title="Alternative metrics × Clinical variables (Spearman r)",
    save_path=os.path.join(FIGURE_DIR, "advanced", "metrics_heatmap.png"),
)

## 6. TLI subscale analysis (Steiger's Z)

In [ ]:
tli_metrics = [PPL_VAR, f"{MODEL}_ppl_p90", f"{MODEL}_prop_high_ppl"]
tli_vars = ["TLI_Impoverished", "TLI_Disorganization", "TLI_Non_specific"]

for metric in tli_metrics:
    sub = patients[[metric] + tli_vars].dropna()
    if len(sub) < 10:
        continue
    r_imp, _ = spearmanr(sub[metric], sub["TLI_Impoverished"])
    r_dis, _ = spearmanr(sub[metric], sub["TLI_Disorganization"])
    r_id, _ = spearmanr(sub["TLI_Impoverished"], sub["TLI_Disorganization"])
    z, p = steigers_z(r_imp, r_dis, r_id, len(sub))
    print(f"{metric}: r_imp={r_imp:.3f}, r_dis={r_dis:.3f}, Steiger Z={z:.3f}, p={p:.4f}")

## 7. ROC analysis

In [ ]:
# Define clinical impairment targets
patients = df[df['is_patient'] == 1].copy()
patients['GAF_impaired'] = (patients['GAF'] < 60).astype(int)
patients['TLI_dis_any'] = (patients['TLI_Disorganization'] > 0).astype(int)
patients['TLI_imp_any'] = (patients['TLI_Impoverished'] > 0).astype(int)

ppl_roc_metrics = [PPL_VAR, f"{MODEL}_ppl_p90", f"{MODEL}_prop_high_ppl"]
roc_targets = {
    'GAF < 60': 'GAF_impaired',
    'TLI Disorganization > 0': 'TLI_dis_any',
    'TLI Impoverished > 0': 'TLI_imp_any',
}

all_rocs = {}
for target_label, target_col in roc_targets.items():
    rocs = compute_roc(patients, ppl_roc_metrics, target_col)
    all_rocs[target_label] = rocs
    for r in rocs:
        print(f"  {target_label} | {r['metric']}: AUC={r['auc']:.3f}")

plot_roc_curves(
    [],  # not used directly
    all_rocs,
    save_path=os.path.join(FIGURE_DIR, "advanced", "roc_curves.png"),
)

In [ ]:
# Combined metric: logistic regression CV
features_combined = [PPL_VAR, f"{MODEL}_ppl_std", f"{MODEL}_prop_high_ppl"]
for target_label, target_col in roc_targets.items():
    auc_combined = logistic_cv_auc(patients, features_combined, target_col)
    auc_singles = {m: logistic_cv_auc(patients, [m], target_col) for m in features_combined}
    print(f"{target_label}: combined AUC={auc_combined:.3f}")
    for m, a in auc_singles.items():
        print(f"  single {m}: AUC={a:.3f}")

## 8. Confounder analysis

In [ ]:
# Confounder visualisation
plot_advanced(
    df=df,
    patients=patients,
    ppl_var=PPL_VAR,
    wc_var=WC_VAR,
    figure_dir=FIGURE_DIR,
)

### 8b. Deconfounding: residualize PPL, then re-run group comparisons & clinical correlations

Instead of nested ANOVA (where confounders absorb the group signal), we regress out
confounders (log word count + education) from perplexity and then test the **residualized**
perplexity for group differences and clinical associations.

In [ ]:
# ── Residualize perplexity ────────────────────────────────────────
# Regress out log(word_count) + education from PPL, keep the residuals.
import statsmodels.api as sm

df_deconf = df.copy()
df_deconf["log_words"] = np.log1p(df_deconf[WC_VAR])

confounders = ["log_words"]
if "education" in df_deconf.columns:
    confounders.append("education")

df_deconf[f"{PPL_VAR}_resid"] = np.nan
resid = residualize_ppl(df_deconf, PPL_VAR, confounders)
df_deconf.loc[resid.index, f"{PPL_VAR}_resid"] = resid

RESID_VAR = f"{PPL_VAR}_resid"
print(f"Confounders removed: {confounders}")
print(f"Residualized variable: {RESID_VAR}")
print(f"N with residuals: {df_deconf[RESID_VAR].notna().sum()}")

# ── Group comparison: raw vs residualized ────────────────────────
print("\n" + "=" * 60)
print("GROUP COMPARISON — Raw PPL")
print("=" * 60)
comp_raw = group_comparison(df_deconf, PPL_VAR)
for k, v in comp_raw.items():
    print(f"  {k}: {v}")

print("\n" + "=" * 60)
print("GROUP COMPARISON — Residualized PPL")
print("=" * 60)
comp_resid = group_comparison(df_deconf.dropna(subset=[RESID_VAR]), RESID_VAR)
for k, v in comp_resid.items():
    print(f"  {k}: {v}")

# Summary table
comp_summary = pd.DataFrame([
    {"Metric": "Raw PPL", **{k: v for k, v in comp_raw.items() if k != "Variable"}},
    {"Metric": "Residualized PPL", **{k: v for k, v in comp_resid.items() if k != "Variable"}},
])
comp_summary

In [ ]:
# ── Clinical correlations: raw vs residualized ───────────────────
patients_deconf = df_deconf[df_deconf["is_patient"] == 1].copy()

print("CLINICAL CORRELATIONS — Raw PPL (patients only)")
corr_raw = clinical_correlations(patients_deconf, PPL_VAR, clinical_vars)
print(corr_raw.to_string(index=False))

print("\nCLINICAL CORRELATIONS — Residualized PPL (patients only)")
corr_resid = clinical_correlations(
    patients_deconf.dropna(subset=[RESID_VAR]), RESID_VAR, clinical_vars
)
print(corr_resid.to_string(index=False))

# Side-by-side comparison
merged_corr = corr_raw.merge(
    corr_resid, on="Clinical Variable", suffixes=("_raw", "_resid")
)
merged_corr

In [ ]:
# ── Visualisation: raw vs residualized ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: Raw PPL
ax = axes[0, 0]
for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
    vals = df_deconf[df_deconf["is_patient"] == is_pat][PPL_VAR].dropna()
    ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
ax.set_title(f"Raw {PPL_VAR}\np(MW)={comp_raw['p(MW)']:.4f}, d={comp_raw['Cohens d']:.3f}")
ax.set_ylabel("PPL"); ax.grid(alpha=0.3)

# Row 1: Residualized PPL
ax = axes[0, 1]
for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
    vals = df_deconf[df_deconf["is_patient"] == is_pat][RESID_VAR].dropna()
    parts = ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
    for pc in parts["bodies"]:
        pc.set_facecolor(clr); pc.set_alpha(0.6)
ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
ax.set_title(f"Residualized PPL\np(MW)={comp_resid['p(MW)']:.4f}, d={comp_resid['Cohens d']:.3f}")
ax.set_ylabel("Residual PPL"); ax.grid(alpha=0.3)

# Row 1: Cohen's d comparison
ax = axes[0, 2]
ax.bar(["Raw", "Residualized"], [comp_raw["Cohens d"], comp_resid["Cohens d"]],
       color=["#3498DB", "#E74C3C"], alpha=0.8)
ax.set_ylabel("Cohen's d"); ax.set_title("Effect size: HC vs Patient")
ax.grid(axis="y", alpha=0.3); ax.axhline(0, color="black", lw=0.5)

# Row 2: Clinical correlations comparison
ax = axes[1, 0]
x = np.arange(len(clinical_vars))
w = 0.35
bars_raw = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_raw"]
bars_res = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_resid"]
ax.bar(x - w/2, bars_raw, w, label="Raw", color="#3498DB", alpha=0.8)
ax.bar(x + w/2, bars_res, w, label="Residualized", color="#E74C3C", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(clinical_vars, rotation=30, ha="right")
ax.set_ylabel("Spearman r"); ax.set_title("Clinical correlations (patients)")
ax.legend(); ax.axhline(0, color="black", lw=0.5); ax.grid(axis="y", alpha=0.3)

# Row 2: Scatter — residualized PPL vs TLI Impoverished
ax = axes[1, 1]
sub = patients_deconf[[RESID_VAR, "TLI_Impoverished", "diagnosis"]].dropna()
if len(sub) >= 5:
    from functions.plotting.plots import DIAG_ORDER, DIAG_COLORS
    diags = [d for d in DIAG_ORDER if d in sub["diagnosis"].values]
    for dx in diags:
        s = sub[sub["diagnosis"] == dx]
        ax.scatter(s[RESID_VAR], s["TLI_Impoverished"], color=DIAG_COLORS.get(dx, "#888"),
                   label=dx, alpha=0.6, s=25)
    r, p = spearmanr(sub[RESID_VAR], sub["TLI_Impoverished"])
    z = np.polyfit(sub[RESID_VAR], sub["TLI_Impoverished"], 1)
    xs = np.linspace(sub[RESID_VAR].min(), sub[RESID_VAR].max(), 100)
    ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
    ax.set_xlabel("Residualized PPL"); ax.set_ylabel("TLI Impoverished")
    ax.set_title(f"Resid PPL vs TLI Imp (r={r:.3f}, p={p:.4f})")
    ax.legend(); ax.grid(alpha=0.2)

# Row 2: Scatter — residualized PPL vs GAF
ax = axes[1, 2]
sub = patients_deconf[[RESID_VAR, "GAF", "diagnosis"]].dropna()
if len(sub) >= 5:
    for dx in diags:
        s = sub[sub["diagnosis"] == dx]
        ax.scatter(s[RESID_VAR], s["GAF"], color=DIAG_COLORS.get(dx, "#888"),
                   label=dx, alpha=0.6, s=25)
    r, p = spearmanr(sub[RESID_VAR], sub["GAF"])
    z = np.polyfit(sub[RESID_VAR], sub["GAF"], 1)
    xs = np.linspace(sub[RESID_VAR].min(), sub[RESID_VAR].max(), 100)
    ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
    ax.set_xlabel("Residualized PPL"); ax.set_ylabel("GAF")
    ax.set_title(f"Resid PPL vs GAF (r={r:.3f}, p={p:.4f})")
    ax.legend(); ax.grid(alpha=0.2)

fig.suptitle("Deconfounded analysis: raw vs residualized perplexity", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "advanced", "deconfounded_analysis.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Per-diagnosis deconfounded comparison vs HC ──────────────────
from functions.analysis.statistics import pairwise_vs_hc, kruskal_wallis_test

print("PAIRWISE vs HC — Raw PPL")
pw_raw = pairwise_vs_hc(df_deconf, PPL_VAR)
print(pd.DataFrame(pw_raw).to_string(index=False))

print("\nPAIRWISE vs HC — Residualized PPL")
pw_resid = pairwise_vs_hc(df_deconf.dropna(subset=[RESID_VAR]), RESID_VAR)
print(pd.DataFrame(pw_resid).to_string(index=False))

print("\nKruskal-Wallis across diagnoses:")
kw_raw_stat, kw_raw_p = kruskal_wallis_test(df_deconf, PPL_VAR)
kw_res_stat, kw_res_p = kruskal_wallis_test(
    df_deconf.dropna(subset=[RESID_VAR]), RESID_VAR
)
print(f"  Raw:          H={kw_raw_stat:.3f}, p={kw_raw_p:.4f}")
print(f"  Residualized: H={kw_res_stat:.3f}, p={kw_res_p:.4f}")

## 9. Summary

In [ ]:
print("Analysis complete.")
print(f"Figures saved to: {os.path.abspath(FIGURE_DIR)}")